# Pipeline timing benchmark

Run this notebook once on a GPU runtime and once on a CPU runtime. It benchmarks one fixed full-field PNG image for `N = 10` measured runs after `N_WARMUP = 3` warmup runs, both with and without classifier filtering. The output CSV is intended for the Sphinx results page and the report timing table.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/satellite_trails
!pip install -e .[dev]


In [ ]:
# Copy the benchmark PNG to Colab local storage before timing.
!rm -rf /content/test_images && mkdir -p /content/test_images && cp /content/drive/MyDrive/satellite_trails/data/test_images/ML1_20190121_194039_red.fits_full.png /content/test_images/


In [ ]:
from pathlib import Path
import sys
from time import perf_counter

import pandas as pd
import torch

PROJECT_ROOT = Path('/content/drive/MyDrive/satellite_trails')
SRC_ROOT = PROJECT_ROOT / 'src'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from segmentation import SatelliteTrailPipeline
from scripts.python.evaluate_final_full_field import build_model
from satellite_trail_segmentation.classifier_model.classifier import TrailClassifier
from satellite_trail_segmentation.unet_model.unet import UNet

SOURCE_PATH = Path('/content/test_images/ML1_20190121_194039_red.fits_full.png')
UNET_CHECKPOINT = PROJECT_ROOT / 'results/models/unet/unet_weights.pt'
CLASSIFIER_CHECKPOINT = PROJECT_ROOT / 'results/models/classifier/classifier_weights.pt'
OUTPUT_CSV = PROJECT_ROOT / 'results/metrics/timing/pipeline_timing_summary.csv'

N = 10
N_WARMUP = 3
PATCH_DIM = 528
NORMALIZATION = 'source_zscore'
UNET_THRESHOLD = 0.65
CLASSIFIER_THRESHOLD = 0.725
NUM_WORKERS = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
runtime_label = 'gpu' if device.type == 'cuda' else 'cpu'
print(f'Benchmark runtime: {runtime_label} ({device})')

In [ ]:
import os
import platform


def _cpu_model_name():
    try:
        with open('/proc/cpuinfo') as file:
            for line in file:
                if line.startswith('model name'):
                    return line.split(':', 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or platform.machine()


def _total_ram_gb():
    try:
        with open('/proc/meminfo') as file:
            for line in file:
                if line.startswith('MemTotal:'):
                    return round(int(line.split()[1]) / 1024**2, 2)
    except OSError:
        pass
    return None


ram_total_gb = _total_ram_gb()
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
gpu_total_memory_gb = (
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
    if torch.cuda.is_available()
    else 0.0
)
hardware_info = {
    'hardware_accelerator': gpu_name if torch.cuda.is_available() else 'CPU',
    'gpu_name': gpu_name,
    'gpu_total_memory_gb': gpu_total_memory_gb,
    'cpu_model': _cpu_model_name(),
    'cpu_count': os.cpu_count(),
    'ram_total_gb': ram_total_gb,
    'high_ram_runtime': bool(ram_total_gb is not None and ram_total_gb >= 40),
}

print('Hardware summary')
for key, value in hardware_info.items():
    print(f'{key}: {value}')


In [ ]:
unet_model = build_model(
    UNet,
    UNET_CHECKPOINT,
    ('in_channels', 'out_channels', 'kernel_size', 'base_channels', 'dropout', 'use_batchnorm'),
)
classifier_model = build_model(
    TrailClassifier,
    CLASSIFIER_CHECKPOINT,
    ('in_channels', 'kernel_size', 'base_channels', 'dropout'),
)

pipeline = SatelliteTrailPipeline(
    unet_model,
    classifier_model=classifier_model,
    patch_dim=PATCH_DIM,
    device=device,
    num_workers=NUM_WORKERS,
)

if not SOURCE_PATH.exists():
    raise FileNotFoundError(f'Missing benchmark image: {SOURCE_PATH}')

benchmark_row = {
    'image_name': SOURCE_PATH.name,
    'image_path': SOURCE_PATH,
}
benchmark_row

In [ ]:
def synchronize():
    if device.type == 'cuda':
        torch.cuda.synchronize()


def run_pipeline_once(use_classifier):
    synchronize()
    total_start = perf_counter()

    preprocess_start = perf_counter()
    preprocessing = pipeline.preprocessing(SOURCE_PATH, normalization=NORMALIZATION)
    preprocessing_time = perf_counter() - preprocess_start

    synchronize()
    segmentation_start = perf_counter()
    prediction, segmentation_times = pipeline.segmentation(
        preprocessing['patch_data'],
        use_classifier=use_classifier,
        unet_threshold=UNET_THRESHOLD,
        classifier_threshold=CLASSIFIER_THRESHOLD,
    )
    synchronize()
    segmentation_time = perf_counter() - segmentation_start

    postprocess_start = perf_counter()
    _, postprocess_times, _ = pipeline.postprocessing(
        prediction['segmented_result'],
        mode='asta_only',
    )
    postprocessing_time = perf_counter() - postprocess_start

    total_time = perf_counter() - total_start

    return {
        'runtime': runtime_label,
        'device': str(device),
        **hardware_info,
        'use_classifier': use_classifier,
        'image_name': SOURCE_PATH.name,
        'preprocessing_time': preprocessing_time,
        'segmentation_time': segmentation_time,
        'classifier_time': segmentation_times.get('classifier_time', 0.0),
        'unet_time': segmentation_times.get('unet_time', 0.0),
        'data_loader_time': segmentation_times.get('init_data_loaders', 0.0),
        'postprocessing_time': postprocessing_time,
        'hough_transform_time': postprocess_times.get('hough_transform', 0.0),
        'total_time': total_time,
    }


def benchmark(use_classifier):
    for _ in range(N_WARMUP):
        run_pipeline_once(use_classifier=use_classifier)

    records = []
    for index in range(1, N + 1):
        print(f"{runtime_label} use_classifier={use_classifier} run {index}/{N} {SOURCE_PATH.name}")
        records.append(run_pipeline_once(use_classifier=use_classifier))
    return records

In [ ]:
records = []
records.extend(benchmark(use_classifier=False))
records.extend(benchmark(use_classifier=True))

raw_timing = pd.DataFrame(records)
raw_timing

In [ ]:
hardware_columns = [
    'hardware_accelerator',
    'gpu_name',
    'gpu_total_memory_gb',
    'cpu_model',
    'cpu_count',
    'ram_total_gb',
    'high_ram_runtime',
]

timing_columns = [
    'preprocessing_time',
    'data_loader_time',
    'classifier_time',
    'unet_time',
    'segmentation_time',
    'postprocessing_time',
    'hough_transform_time',
    'total_time',
]

summary = (
    raw_timing
    .groupby(['runtime', 'device', *hardware_columns, 'use_classifier'])[timing_columns]
    .agg(['mean', 'std'])
    .reset_index()
)
summary.columns = ['_'.join(col).rstrip('_') if isinstance(col, tuple) else col for col in summary.columns]
summary.insert(len(['runtime', 'device', *hardware_columns, 'use_classifier']), 'n', N)
summary.insert(len(['runtime', 'device', *hardware_columns, 'use_classifier']) + 1, 'n_warmup', N_WARMUP)

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
if OUTPUT_CSV.exists():
    existing = pd.read_csv(OUTPUT_CSV)
    summary = pd.concat([existing, summary], ignore_index=True)
    summary = summary.drop_duplicates(subset=['runtime', 'use_classifier'], keep='last')

summary.to_csv(OUTPUT_CSV, index=False)
summary

After running the notebook on both CPU and GPU runtimes, `results/metrics/timing/pipeline_timing_summary.csv` should contain four rows: CPU without classifier, CPU with classifier, GPU without classifier, and GPU with classifier. Each row summarizes ten measured runs on the same fixed test image.

In [ ]:
from google.colab import runtime
runtime.unassign()
